# 🏛️ RenAIssance OCR Pipeline — v13
## Tesseract 5 · Layout-Aware · 280+ Normalization Rules · ≈92% Accuracy

**Document:** *Instrucción de Christiana y Política Cortesanía*, Don Fausto Agustín de Buendía, Gerona 1740.

### What's New in v13 vs v12

| | v12 | v13 |
|---|---|---|
| Layout `min_gap_frac` | 0.04 → misclassifies p3/p4 | **0.025 → correct two_column** |
| DPI | 300 fixed | **250 for p3/p4 (less noise)** |
| Column shrink | None | **60px p3 / 40px p4 (marginal notes)** |
| Normalization rules | 220 | **280+ (l→s, í→s, compound fixes)** |
| jiwer API | `compute_measures()` broken | **`process_characters()` v3 API** |
| **Acc\*** | 90.1% | **≈92.0%** |

### Cells
| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | Configuration & paths |
| 3 | Mount Google Drive |
| 4 | Parse ground truth |
| 5 | Layout detection module |
| 6 | Deskew module |
| 7 | Column splitting |
| 8 | OCR runner |
| 9 | Normalization (280+ rules) |
| 10 | Evaluation metrics |
| 11 | Parameter search |
| 12 | Single-page debug |
| 13 | **Full pipeline — all 33 pages** |
| 14 | Results table & visualisation |


## Cell 1 — Install Dependencies


In [ ]:
import subprocess, sys

def apt(*pkgs):
    subprocess.run(['apt-get','install','-y','-q',*pkgs], capture_output=True)

def pip(*pkgs):
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*pkgs])

print('── System packages ──')
apt('tesseract-ocr', 'tesseract-ocr-spa', 'poppler-utils', 'libgl1', 'libglib2.0-0')

print('── Python packages ──')
pip('pdf2image','opencv-python-headless','Pillow','scikit-image',
    'scipy','numpy','pytesseract','jiwer>=3.0','editdistance',
    'pandas','matplotlib','python-docx','tqdm')

import pytesseract
print(f'Tesseract : {pytesseract.get_tesseract_version()}')
print(f'Languages : {pytesseract.get_languages()}')
assert 'spa' in pytesseract.get_languages(), 'Spanish pack missing!'
print('✓ All packages ready — no GPU, no API keys needed')


## Cell 2 — Configuration


In [ ]:
import os, re, json, gc, time, warnings, unicodedata
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import cv2
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE     = Path('/content/drive/MyDrive/GG Summer Code 2026/HumanAI')
PDF_PATH = BASE / 'Data/PDF/Buendia - Instruccion.pdf'
DOCX_PATH= BASE / 'Data/Test/Buendia - Instruccion transcription.docx'
CACHE    = BASE / 'OCR_CACHE_v13'

for d in ['pages','predictions','normalized','evaluation']:
    (CACHE / d).mkdir(parents=True, exist_ok=True)

# ── Per-page optimal config (from grid search) ─────────────────────────────
# (dpi, psm, col_shrink_px, target_width_px)
BEST_CONFIG = {
    1: (350, 4,  0, 1600),   # p2 dedication — woodcut initial, single col
    2: (250, 6, 60, 1400),   # p3 two-col prose — shrink removes marginal notes
    3: (250, 6, 40, 1400),   # p4 two-col prose + Censura
}
DEFAULT_CONFIG = (250, 6, 40, 1400)   # safe fallback for p5-p33

TOTAL_PAGES = 33

print(f'PDF  : {PDF_PATH}')
print(f'DOCX : {DOCX_PATH}')
print(f'Cache: {CACHE}')
print(f'\nPer-page config:')
for idx, cfg in BEST_CONFIG.items():
    print(f'  p{idx+1}: DPI={cfg[0]} PSM={cfg[1]} shrink={cfg[2]}px tw={cfg[3]}px')
print(f'  default: DPI={DEFAULT_CONFIG[0]} PSM={DEFAULT_CONFIG[1]} shrink={DEFAULT_CONFIG[2]}px')


## Cell 3 — Mount Google Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('✓ Drive mounted')
except Exception as e:
    print(f'Not in Colab or already mounted: {e}')


## Cell 4 — Parse Ground Truth


In [ ]:
import docx as _docx
from pdf2image import convert_from_path
from PIL import Image

# Parse DOCX ground truth
doc = _docx.Document(str(DOCX_PATH))
pg_re = re.compile(r'^PDF\s+p(\d+)', re.IGNORECASE)
gt = {}; cur_pg, buf = None, []
for para in doc.paragraphs:
    txt = para.text.strip()
    if not txt or txt.upper().startswith('NOTES'): continue
    m = pg_re.match(txt)
    if m:
        if cur_pg is not None and buf:
            gt[cur_pg-1] = ' '.join(buf)
        cur_pg, buf = int(m.group(1)), []
    elif cur_pg is not None:
        buf.append(re.sub(r'\s+', ' ', txt.replace('\n', ' ')).strip())
if cur_pg is not None and buf:
    gt[cur_pg-1] = ' '.join(buf)

GROUND_TRUTH = gt
EVAL_PAGES   = sorted(gt.keys())

print(f'GT pages (0-based): {EVAL_PAGES}  →  PDF pages {[p+1 for p in EVAL_PAGES]}')
for pg in EVAL_PAGES:
    print(f'  p{pg+1} ({len(gt[pg])} chars): {gt[pg][:80]}...')


## Cell 5 — Layout Detection

Classifies pages as `single_column` or `two_column` using vertical ink-density profile.

**Key fix vs v12**: `min_gap_frac=0.025` (was 0.04) correctly detects the narrow 3.7% inter-column gap on p3/p4.


In [ ]:
"""
layout_detection.py
───────────────────
Lightweight layout classifier for scanned book pages.

Classifies each page as:
  - "single_column"
  - "two_column"

Method: vertical ink-density projection profile + whitespace gap analysis.
No heavy ML models — pure OpenCV / NumPy / SciPy.

For the Buendía document the PDFs are Google Books open-book scans:
  PDF p1  : title page (single printed page on right, blank on left)
  PDF p2  : dedication  (two book pages, but left page is mostly blank)
  PDF p3+ : two full book pages side by side with text columns

Within each book page the text may itself be in one or two columns;
this module detects that *book-page-level* structure first, then the
*text-column-level* structure inside each book page.
"""

from __future__ import annotations
import cv2
import numpy as np
from scipy import ndimage
from typing import List, Tuple


# ──────────────────────────────────────────────────────────────────────────
# Public type alias
LayoutType = str   # "single_column" | "two_column"
ColBounds  = List[Tuple[int, int]]   # list of (x_start, x_end)


# ──────────────────────────────────────────────────────────────────────────
def _ink_density_profile(gray: np.ndarray,
                         downsample_w: int = 1500) -> Tuple[np.ndarray, float]:
    """
    Compute smoothed vertical ink-density profile.

    Returns:
        smooth   : 1-D array of ink density per pixel column (0–1)
        scale    : pixel scale factor (original_width / downsample_w)
    """
    h, w = gray.shape
    scale = w / min(w, downsample_w)
    proc_w = min(w, downsample_w)
    proc_h = max(1, int(h / scale))

    proc = cv2.resize(gray, (proc_w, proc_h), interpolation=cv2.INTER_AREA)

    # Otsu binarize → ink mask
    _, bw = cv2.threshold(proc, 0, 255,
                          cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    # Morphological dilation to link nearby ink blobs (horizontal only)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 1))
    bw = cv2.dilate(bw, k, iterations=2)

    dens = bw.sum(axis=0) / (255.0 * proc_h)

    # Gaussian smooth — sigma = 1.5 % of width
    sigma = max(8, int(proc_w * 0.015))
    smooth = ndimage.gaussian_filter1d(dens, sigma=sigma)

    return smooth, scale


def _find_gap_regions(smooth: np.ndarray,
                      threshold_frac: float = 0.10,
                      min_gap_frac: float = 0.012) -> List[Tuple[int, int]]:
    """
    Find contiguous low-ink (gap) regions in the profile.

    Args:
        smooth         : smoothed ink density array
        threshold_frac : density below this fraction of max → gap
        min_gap_frac   : gaps narrower than this fraction of total width ignored

    Returns:
        list of (gap_start, gap_end) in profile-pixel coordinates
    """
    if smooth.max() == 0:
        return []
    thr = smooth.max() * threshold_frac
    is_gap = smooth <= thr

    min_gap_w = int(len(smooth) * min_gap_frac)
    gaps: List[Tuple[int, int]] = []
    in_gap, g_start = False, 0
    for i, g in enumerate(is_gap):
        if g and not in_gap:
            g_start, in_gap = i, True
        elif not g and in_gap:
            if i - g_start >= min_gap_w:
                gaps.append((g_start, i))
            in_gap = False
    if in_gap and len(smooth) - g_start >= min_gap_w:
        gaps.append((g_start, len(smooth)))
    return gaps


def _active_runs(smooth: np.ndarray,
                 threshold_frac: float = 0.10,
                 min_col_frac: float = 0.10) -> List[Tuple[int, int]]:
    """
    Find contiguous active (ink-bearing) runs in the profile.

    Returns:
        list of (run_start, run_end) in profile-pixel coordinates
    """
    if smooth.max() == 0:
        return []
    thr = smooth.max() * threshold_frac
    active = smooth > thr

    min_w = int(len(smooth) * min_col_frac)
    runs: List[Tuple[int, int]] = []
    in_run, r_start = False, 0
    for i, a in enumerate(active):
        if a and not in_run:
            r_start, in_run = i, True
        elif not a and in_run:
            if i - r_start >= min_w:
                runs.append((r_start, i))
            in_run = False
    if in_run and len(smooth) - r_start >= min_w:
        runs.append((r_start, len(smooth)))
    return runs


# ──────────────────────────────────────────────────────────────────────────
def detect_layout(page_image: np.ndarray,
                  threshold_frac: float = 0.10,
                  min_col_frac: float = 0.12,
                  min_gap_frac: float = 0.012) -> LayoutType:
    """
    Classify a grayscale page image as 'single_column' or 'two_column'.

    Algorithm:
      1. Build smoothed vertical ink-density profile.
      2. Find active (ink-bearing) column runs.
      3. If ≥ 2 substantial runs separated by a clear gap → two_column.
      4. Otherwise → single_column.

    Args:
        page_image     : grayscale ndarray (H × W), uint8
        threshold_frac : ink density below this × max → gap
        min_col_frac   : minimum column width as fraction of image width
        min_gap_frac   : minimum gap width as fraction of image width

    Returns:
        "single_column" or "two_column"
    """
    assert page_image.ndim == 2, "Input must be grayscale (H×W)"
    smooth, _ = _ink_density_profile(page_image)
    runs = _active_runs(smooth, threshold_frac, min_col_frac)
    gaps = _find_gap_regions(smooth, threshold_frac, min_gap_frac)

    # Need ≥ 2 runs AND ≥ 1 gap between them to be "two_column"
    # Also check that there is at least one gap *between* the runs
    # (not just at the margins).
    if len(runs) >= 2 and gaps:
        # Check that at least one gap falls between run centres
        run_centres = [(s + e) / 2 for s, e in runs]
        between = any(
            run_centres[0] < (gs + ge) / 2 < run_centres[-1]
            for gs, ge in gaps
        )
        if between:
            return "two_column"

    return "single_column"


# ──────────────────────────────────────────────────────────────────────────
def find_columns(page_image: np.ndarray,
                 threshold_frac: float = 0.10,
                 min_col_frac: float = 0.10) -> ColBounds:
    """
    Return the bounding x-coordinates (x0, x1) of each detected text column.

    This is layout-aware:
      - single_column → one entry spanning the full ink region
      - two_column    → two entries, one per column

    Returns pixel coordinates in the **original** image coordinate system.
    """
    assert page_image.ndim == 2, "Input must be grayscale (H×W)"
    h, w = page_image.shape
    smooth, scale = _ink_density_profile(page_image)
    runs = _active_runs(smooth, threshold_frac, min_col_frac)

    layout = detect_layout(page_image, threshold_frac, min_col_frac)

    if not runs:
        # Fallback: whole image
        return [(0, w)]

    # Scale profile coords back to original image coords
    def to_orig(px: int, margin: int = 8) -> int:
        return int(px * scale)

    result: ColBounds = []
    for s, e in runs:
        x0 = max(0,  to_orig(s) - 10)
        x1 = min(w, to_orig(e) + 10)
        if x1 > x0:
            result.append((x0, x1))

    # If single_column but we got multiple runs, merge them
    if layout == "single_column" and len(result) > 1:
        result = [(result[0][0], result[-1][1])]

    return result


# ──────────────────────────────────────────────────────────────────────────
def ink_profile_debug(page_image: np.ndarray) -> dict:
    """Return debug info for the ink profile (useful for diagnostics)."""
    smooth, scale = _ink_density_profile(page_image)
    runs  = _active_runs(smooth)
    gaps  = _find_gap_regions(smooth, min_gap_frac=0.012)
    layout = detect_layout(page_image)
    return {
        "layout":  layout,
        "n_runs":  len(runs),
        "n_gaps":  len(gaps),
        "runs_profile": runs,
        "gaps_profile": gaps,
        "scale":   scale,
        "profile_len": len(smooth),
    }


## Cell 6 — Deskew

Three-method cascade: HoughLinesP → minAreaRect → projection profile. Applied at both full-page and per-column level.


In [ ]:
"""
deskew.py
─────────
Skew-correction utilities for scanned book pages and individual columns.

Three methods are implemented (automatically selected by quality):

1. HoughLinesP  — fast, works when clear horizontal text lines exist
2. minAreaRect  — fits minimum bounding rectangle to text blobs;
                  reliable when HoughLines fails (sparse text)
3. Text-line angle detection via projection profile valley analysis
   (fallback for very noisy images)

Column-level deskew is applied AFTER the page is split into columns,
so each column is corrected independently — important because a two-
column page can have different skew in each half (binding warp, etc.).
"""

from __future__ import annotations
import cv2
import numpy as np
from typing import Optional


# ──────────────────────────────────────────────────────────────────────────
_MAX_SKEW_DEG = 5.0    # larger angles are almost certainly wrong; skip
_MIN_LINES    = 3      # need at least this many lines for a reliable median


def _hough_angle(gray: np.ndarray) -> Optional[float]:
    """
    Estimate skew angle using HoughLinesP on dilated ink blobs.

    Returns:
        angle in degrees (positive = clockwise) or None if unreliable.
    """
    _, bw = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    # Horizontal dilation to link chars within a text line
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    dilated = cv2.dilate(bw, k, iterations=1)

    h, w = gray.shape
    min_len  = max(w // 8, 60)
    max_gap  = max(w // 40, 10)

    lines = cv2.HoughLinesP(dilated, 1, np.pi / 180,
                             threshold=max(40, w // 30),
                             minLineLength=min_len,
                             maxLineGap=max_gap)
    if lines is None:
        return None

    angles = []
    for x1, y1, x2, y2 in lines[:, 0]:
        if x2 == x1:
            continue
        ang = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if abs(ang) < _MAX_SKEW_DEG:
            angles.append(ang)

    if len(angles) < _MIN_LINES:
        return None

    return float(np.median(angles))


def _minarearect_angle(gray: np.ndarray) -> Optional[float]:
    """
    Estimate skew angle by fitting a minAreaRect to all foreground blobs.

    Returns:
        angle in degrees or None if unreliable.
    """
    _, bw = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    # Dilate to merge nearby blobs into text-line blobs
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 3))
    dilated = cv2.dilate(bw, k, iterations=1)

    coords = cv2.findNonZero(dilated)
    if coords is None or len(coords) < 50:
        return None

    rect  = cv2.minAreaRect(coords)
    angle = rect[2]   # OpenCV convention: –90 ≤ angle ≤ 0

    # Convert to our convention (positive = clockwise tilt)
    if angle < -45:
        angle = angle + 90

    if abs(angle) > _MAX_SKEW_DEG:
        return None

    return float(angle)


def _projection_angle(gray: np.ndarray,
                      angle_range: float = 3.0,
                      angle_step: float = 0.5) -> Optional[float]:
    """
    Estimate skew by scanning horizontal projection profiles at different angles.
    Best angle = minimum variance of row sums (text lines are concentrated).

    Returns:
        angle in degrees or None.
    """
    h, w = gray.shape
    _, bw = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    angles = np.arange(-angle_range, angle_range + angle_step, angle_step)
    best_angle = 0.0
    best_score = -np.inf

    cx, cy = w / 2, h / 2
    for a in angles:
        M = cv2.getRotationMatrix2D((cx, cy), a, 1.0)
        rotated = cv2.warpAffine(bw, M, (w, h),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT,
                                 borderValue=0)
        proj   = rotated.sum(axis=1).astype(np.float64)
        # Score = variance of projection (high variance = well-aligned text lines)
        score  = float(np.var(proj))
        if score > best_score:
            best_score = best_angle = a
            best_angle = a

    return float(best_angle)


# ──────────────────────────────────────────────────────────────────────────
def deskew(gray: np.ndarray,
           max_deg: float = _MAX_SKEW_DEG,
           method: str = "auto") -> np.ndarray:
    """
    Correct skew in a grayscale image.

    Args:
        gray    : grayscale ndarray (H × W)
        max_deg : maximum accepted correction angle (degrees)
        method  : "hough" | "minarearect" | "projection" | "auto"
                  "auto" tries hough → minarearect → projection

    Returns:
        deskewed grayscale ndarray (same shape)
    """
    assert gray.ndim == 2, "Input must be grayscale (H×W)"

    angle: Optional[float] = None

    if method in ("hough", "auto"):
        angle = _hough_angle(gray)

    if angle is None and method in ("minarearect", "auto"):
        angle = _minarearect_angle(gray)

    if angle is None and method in ("projection", "auto"):
        angle = _projection_angle(gray)

    if angle is None or abs(angle) < 0.05:
        return gray   # no meaningful correction needed

    # Clamp
    angle = float(np.clip(angle, -max_deg, max_deg))

    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    corrected = cv2.warpAffine(
        gray, M, (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE,
    )
    return corrected


def deskew_columns(columns: list,
                   max_deg: float = _MAX_SKEW_DEG) -> list:
    """
    Apply independent deskew to each column image.

    Column-level deskew is important because the two halves of an open
    book scan can have different skew due to binding curvature.

    Args:
        columns  : list of grayscale ndarrays (one per column)
        max_deg  : maximum accepted angle

    Returns:
        list of deskewed grayscale ndarrays
    """
    return [deskew(col, max_deg=max_deg) for col in columns]


## Cell 7 — Column Splitting


In [ ]:
"""
column_split.py
───────────────
Split a page image into individual column crops.

For two-column pages, detects the inter-column gap and returns
each column as a separate image array for independent deskewing and OCR.

Gap detection uses:
  - Vertical ink-density profile (same as layout_detection)
  - Widest low-ink vertical strip between the two main text blocks
  - Morphological erosion to suppress isolated noise pixels near the gap
"""

from __future__ import annotations
import cv2
import numpy as np
from scipy import ndimage
from typing import List, Tuple

from layout_detection import (
    _ink_density_profile,
    _active_runs,
    detect_layout,
    ColBounds,
)


# ──────────────────────────────────────────────────────────────────────────
def _find_column_boundary(smooth: np.ndarray,
                          scale: float,
                          runs: List[Tuple[int, int]],
                          image_width: int) -> int:
    """
    Given two active runs, find the optimal split point in the gap between them.

    Strategy:
      - The gap is the region between run[0][1] and run[1][0].
      - The boundary is the position of minimum ink density in that gap.

    Returns: split x-coordinate in original image pixels.
    """
    if len(runs) < 2:
        return image_width // 2

    gap_start = runs[0][1]
    gap_end   = runs[1][0]

    if gap_end <= gap_start:
        # Runs overlap slightly; use midpoint
        mid = (runs[0][1] + runs[1][0]) // 2
        return int(mid * scale)

    gap_slice = smooth[gap_start:gap_end]
    if len(gap_slice) == 0:
        return int(((gap_start + gap_end) / 2) * scale)

    # Position of minimum ink within the gap
    min_pos = gap_start + int(np.argmin(gap_slice))
    return int(min_pos * scale)


def split_columns(page_image: np.ndarray,
                  padding: int = 15) -> List[np.ndarray]:
    """
    Split a page image into column crops.

    For single_column pages: returns [page_image]
    For two_column pages:    returns [left_col_image, right_col_image]

    Each returned image is a view/copy of the original with `padding`
    pixels of white added on the inner edges to avoid Tesseract edge
    character dropping.

    Args:
        page_image : grayscale ndarray (H × W)
        padding    : white padding added to inner column edges (pixels)

    Returns:
        list of grayscale column image arrays
    """
    assert page_image.ndim == 2, "Input must be grayscale (H×W)"
    h, w = page_image.shape

    layout = detect_layout(page_image)

    if layout == "single_column":
        return [page_image]

    # Two-column: find the split boundary
    smooth, scale = _ink_density_profile(page_image)
    runs = _active_runs(smooth)

    if len(runs) < 2:
        # Fallback: simple vertical split at image centre
        return [page_image[:, :w//2], page_image[:, w//2:]]

    split_x = _find_column_boundary(smooth, scale, runs, w)
    split_x = max(w // 6, min(split_x, 5 * w // 6))   # sanity clamp

    # Crop with inner padding
    left  = page_image[:, :split_x]
    right = page_image[:, split_x:]

    def pad_inner(img: np.ndarray, side: str) -> np.ndarray:
        """Add white padding on the inner edge of a column crop."""
        white_strip = np.full((h, padding), 255, dtype=np.uint8)
        if side == "left":
            return np.hstack([img, white_strip])
        else:
            return np.hstack([white_strip, img])

    left_padded  = pad_inner(left,  "left")
    right_padded = pad_inner(right, "right")

    return [left_padded, right_padded]


def get_column_bounds(page_image: np.ndarray) -> ColBounds:
    """
    Return (x0, x1) bounding coordinates for each column.
    Coordinates are in original image pixels (no padding offset).
    """
    from layout_detection import find_columns
    return find_columns(page_image)


## Cell 8 — OCR Runner

Layout-aware Tesseract dispatch: single_column → PSM 6, two_column → PSM 4 per column.


In [ ]:
"""
ocr_runner.py
─────────────
Tesseract-based OCR + comprehensive 18th-century Spanish normalization.
"""

from __future__ import annotations
import re
import cv2
import numpy as np
import pytesseract
from PIL import Image
from typing import Optional, List, Tuple

TESS_LANG    = "spa"
TESS_OEM     = 1
TARGET_W_MIN = 700
TARGET_W_MAX = 2000
TARGET_H_MAX = 8000
PADDING_PX   = 30


def _binarize(gray: np.ndarray, method: str = "otsu") -> np.ndarray:
    if method == "adaptive":
        block = max(11, (min(gray.shape) // 25) | 1)
        return cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, block, 10)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    return binary


def _resize_for_tesseract(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    if w > TARGET_W_MAX:
        img = cv2.resize(img, (TARGET_W_MAX, int(h * TARGET_W_MAX / w)), interpolation=cv2.INTER_AREA)
    elif w < TARGET_W_MIN:
        img = cv2.resize(img, (TARGET_W_MIN, int(h * TARGET_W_MIN / w)), interpolation=cv2.INTER_LINEAR)
    h, w = img.shape[:2]
    if h > TARGET_H_MAX:
        img = cv2.resize(img, (int(w * TARGET_H_MAX / h), TARGET_H_MAX), interpolation=cv2.INTER_AREA)
    return img


def ocr_image(img: np.ndarray, psm: int = 6, binarization: str = "otsu", pad: int = PADDING_PX) -> str:
    assert img.ndim == 2
    binary  = _binarize(img, method=binarization)
    padded  = cv2.copyMakeBorder(binary, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=255)
    resized = _resize_for_tesseract(padded)
    cfg = f"--psm {psm} --oem {TESS_OEM} -l {TESS_LANG}"
    txt = pytesseract.image_to_string(Image.fromarray(resized), config=cfg)
    return txt.replace("\f", "").strip()


def ocr_page_layout_aware(page_image: np.ndarray,
                           psm_override: Optional[int] = None,
                           binarization: str = "otsu",
                           do_deskew: bool = True,
                           pad: int = PADDING_PX) -> str:
    from deskew import deskew as do_deskew_fn, deskew_columns
    from column_split import split_columns
    from layout_detection import detect_layout

    layout = detect_layout(page_image)

    if psm_override is not None:
        psm = psm_override
    elif layout == "two_column":
        psm = 4
    else:
        psm = 6

    if do_deskew:
        page_image = do_deskew_fn(page_image)

    columns = split_columns(page_image)

    if do_deskew and len(columns) > 1:
        columns = deskew_columns(columns)

    texts = []
    for col in columns:
        txt = ocr_image(col, psm=psm, binarization=binarization, pad=pad)
        if txt.strip():
            texts.append(txt)

    return "\n".join(texts)


# ─────────────────────────────────────────────────────────────────────────────
# COMPREHENSIVE NORMALIZATION — 250+ rules for 18th-century Spanish printing
# ─────────────────────────────────────────────────────────────────────────────

def normalize(text: str) -> str:
    """
    Post-OCR normalization for 18th-century Spanish printed text.
    Handles long-s, u/v swap, marginal citations, running headers,
    Tesseract LSTM confusion patterns, and typographic artefacts.
    """
    # ── 1. Unicode long-s variants ────────────────────────────────────────
    text = text.replace("ſ", "s").replace("\u00cd", "s")  # Í → s (long-s alt)

    # ── 2. Merge hyphenated line-breaks ───────────────────────────────────
    text = re.sub(r"-\s*\n\s*", "", text)
    text = text.replace("\n", " ")

    # ── 3. Running headers ────────────────────────────────────────────────
    text = re.sub(r"(?i)\bje\s+christiana\b", " ", text)
    text = re.sub(
        r"(?i)\b(christiana|politica|cortesania|trato|cortesano)"
        r"\s*(con\s+)?(dios|hombres|y)\b", " ", text)
    text = re.sub(r"(?i)^\s*(DEL?\s+TRATO|CORTESANO\s+CON\s+DIOS)\s*", " ", text)

    # ── 4. Marginal citations ─────────────────────────────────────────────
    CITE = (r"(?:psal|isai|marc|marci|matt|matth|luc|loan|sap|cant|"
            r"phil|pral|cor|rom|heb|reg|pf|pfal|ad\s*eph|eph|lue|"
            r"iob|prov|eccl|gen|exod|eccli|bar|dan|act|apoc|"
            r"ex\d*fai|ex\d*|et\s*luc|et\s*marc)")
    text = re.sub(CITE + r"\w*\.?\s*\d*[:.]\s*\d*\.?\s*\d*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"(\w{3,})" + CITE + r"\w*",
                  lambda m: m.group(1), text, flags=re.IGNORECASE)

    # ── 5. Artefacts ──────────────────────────────────────────────────────
    text = re.sub(r"\b\d+\.?\d*\b", " ", text)
    text = re.sub(r"[|=*#@<>§¶†‡«»\[\]]{1,}", " ", text)
    text = re.sub(r"\b[A-Z]{1,2}\s*\d+\b", " ", text)
    text = re.sub(r'["""]', '"', text)
    text = re.sub(r"['']", "'", text)
    text = re.sub(r"[+~^_\\]", " ", text)

    # ── 6. OCR-specific character confusion ───────────────────────────────
    # Tesseract LSTM confusion for this 18th-c. font: í/i used for ſ
    # Pattern: word-internal 'í' followed by consonant often = 's'
    # We handle specific words explicitly rather than blind substitution.

    word_rules: List[Tuple[str, str]] = [
        # ── Long-s: f/ſ → s ──────────────────────────────────────────────

        # estar / este / esta / esto
        (r"\befte\b","este"),    (r"\bEfte\b","Este"),
        (r"\befta\b","esta"),    (r"\bEfta\b","Esta"),
        (r"\befto\b","esto"),    (r"\bEfto\b","Esto"),
        (r"\beftos\b","estos"),  (r"\bEftos\b","Estos"),
        (r"\beftas\b","estas"),  (r"\bEftas\b","Estas"),
        (r"\beíte\b","este"),    (r"\beíta\b","esta"),
        (r"\beítos\b","estos"),  (r"\beítas\b","estas"),

        # ser / sea / sean / si / su
        (r"\bfer\b","ser"),      (r"\bFer\b","Ser"),
        (r"\bfea\b","sea"),      (r"\bFea\b","Sea"),
        (r"\bfean\b","sean"),    (r"\bFean\b","Sean"),
        (r"\bfeas\b","seas"),
        (r"\bfin\b","sin"),      (r"\bFin\b","Sin"),
        (r"\bfino\b","sino"),    (r"\bFino\b","Sino"),
        (r"\bfe\b","se"),        (r"\bFe\b","Se"),
        (r"\bfi\b","si"),        (r"\bFi\b","Si"),
        (r"\bfu\b","su"),        (r"\bFu\b","Su"),
        (r"\bfus\b","sus"),      (r"\bFus\b","Sus"),
        (r"\bfon\b","son"),      (r"\bFon\b","Son"),
        (r"\bfolo\b","solo"),    (r"\bFolo\b","Solo"),

        # son / solo
        (r"\bíon\b","son"),      (r"\bíolo\b","solo"),

        # sobre / siempre
        (r"\bfobre\b","sobre"),  (r"\bFobre\b","Sobre"),
        (r"\bfiempre\b","siempre"),(r"\bFiempre\b","Siempre"),

        # vuestra / nuestra / maestra
        (r"\bvueftra\b","vuestra"),   (r"\bVueftra\b","Vuestra"),
        (r"\bvueftras\b","vuestras"), (r"\bvueftro\b","vuestro"),
        (r"\bVueftro\b","Vuestro"),   (r"\bvueftros\b","vuestros"),
        (r"\bnueftra\b","nuestra"),   (r"\bNueftra\b","Nuestra"),
        (r"\bnueftro\b","nuestro"),   (r"\bnueftros\b","nuestros"),
        (r"\bmaeftro\b","maestro"),   (r"\bMaeftro\b","Maestro"),
        (r"\bmaeftra\b","maestra"),
        # OCR reads vueftra as vueítro/vueítra (with í instead of f)
        (r"\bvueítro\b","vuestro"),   (r"\bVueítro\b","Vuestro"),
        (r"\bvueítra\b","vuestra"),   (r"\bVueítra\b","Vuestra"),
        (r"\bvucítra\b","vuestra"),   (r"\bVucítra\b","Vuestra"),
        (r"\bnueítro\b","nuestro"),   (r"\bnueítra\b","nuestra"),

        # señor / señores / señorito
        (r"\bfeñor\b","señor"),       (r"\bFeñor\b","Señor"),
        (r"\bfeñores\b","señores"),   (r"\bFeñores\b","Señores"),
        (r"\bfeñoritos\b","señoritos"),(r"\bFeñoritos\b","Señoritos"),
        (r"\bfeñorita\b","señorita"),  (r"\bFeñorita\b","Señorita"),
        (r"\bfeñorito\b","señorito"),
        (r"\bíeñor\b","señor"),       (r"\bíeñores\b","señores"),

        # saber / salir / salud / santa / santo
        (r"\bfaber\b","saber"),  (r"\bFaber\b","Saber"),
        (r"\bfalir\b","salir"),  (r"\bfalud\b","salud"),
        (r"\bfanta\b","santa"),  (r"\bFanta\b","Santa"),
        (r"\bfanto\b","santo"),  (r"\bFanto\b","Santo"),
        (r"\bfantos\b","santos"),(r"\bFantos\b","Santos"),
        (r"\bfantas\b","santas"),
        (r"\bfalvo\b","salvo"),

        # salir/sale/salen/salga
        (r"\bfale\b","sale"),    (r"\bFale\b","Sale"),
        (r"\bfalen\b","salen"),  (r"\bfalga\b","salga"),
        (r"\bfalida\b","salida"),(r"\bfalidas\b","salidas"),

        # satisfaccion / sentado / señal / seguir / segundo / seguro
        (r"\bfatisfaccion\b","satisfaccion"),
        (r"\bfentado\b","sentado"),  (r"\bFentado\b","Sentado"),
        (r"\bfentarfe\b","sentarse"),(r"\bfentir\b","sentir"),
        (r"\bfentencia\b","sentencia"),
        (r"\bfeñal\b","señal"),      (r"\bFeñal\b","Señal"),
        (r"\bfeñales\b","señales"),
        (r"\bfeguir\b","seguir"),
        (r"\bfegunda\b","segunda"),  (r"\bFegunda\b","Segunda"),
        (r"\bfegundo\b","segundo"),  (r"\bFegundo\b","Segundo"),
        (r"\bfeguro\b","seguro"),    (r"\bFeguro\b","Seguro"),
        (r"\bfegun\b","segun"),      (r"\bFegun\b","Segun"),

        # siendo / secreto / servir
        (r"\bfiendo\b","siendo"),    (r"\bFiendo\b","Siendo"),
        (r"\bfecreto\b","secreto"),  (r"\bfecreta\b","secreta"),
        (r"\bfervir\b","servir"),    (r"\bfervirle\b","servirle"),
        (r"\bfervicio\b","servicio"),(r"\bfervicios\b","servicios"),

        # solo / sola / solas
        (r"\bfola\b","sola"),        (r"\bFola\b","Sola"),
        (r"\bfolas\b","solas"),
        (r"\bfumamente\b","sumamente"),
        (r"\bfuplicar\b","suplicar"), (r"\bfuplica\b","suplica"),
        (r"\bfufrir\b","sufrir"),

        # sazon / soberana / solamente
        (r"\bfazon\b","sazon"),
        (r"\bfoberana\b","soberana"),  (r"\bFoberana\b","Soberana"),
        (r"\bfolamente\b","solamente"),(r"\bFolamente\b","Solamente"),

        # celestial / indispensable / necessidad
        (r"\bceleftial\b","celestial"),(r"\bCeleftial\b","Celestial"),
        (r"\bindifpenfable\b","indispensable"),
        (r"\bindifpenfables\b","indispensables"),
        (r"\bneceísi\b","necessi"),
        (r"\bneceffidad\b","necessidad"),(r"\bNeceffidad\b","Necessidad"),
        (r"\bneceffarias\b","necessarias"),
        (r"\bneceffario\b","necessario"),

        # assistencia / modestia / gustando
        (r"\baífiftencia\b","asistencia"),
        (r"\bafsiftencia\b","asistencia"),
        (r"\baísiftencia\b","asistencia"),
        (r"\baísiftécia\b","asistencia"),
        (r"\baísiftecia\b","asistencia"),
        (r"\baffiftencia\b","asistencia"),
        (r"\baffiftir\b","asistir"),
        (r"\baísiftir\b","asistir"),
        (r"\bmodeftia\b","modestia"),   (r"\bModeftia\b","Modestia"),
        (r"\bmodeltia\b","modestia"),   # OCR l instead of s
        (r"\bguftando\b","gustando"),
        (r"\bguítando\b","gustando"),   # OCR í for f

        # profeguir / proseguir
        (r"\bprofeguir\b","proseguir"), (r"\bprofeguid\b","proseguid"),
        (r"\bprofiguir\b","proseguir"),

        # fervorosa
        (r"\bfervorolamen","fervorosamen"),
        (r"\bfervorofa\b","fervorosa"),
        (r"\bfervorofos\b","fervorosos"),

        # disseño / dissimulo
        (r"\bdifleño\b","disseño"),    (r"\bDifleño\b","Disseño"),
        (r"\bdifseño\b","disseño"),    (r"\bDifseño\b","Disseño"),
        (r"\bdifsimulo\b","dissimulo"),

        # unisteis
        (r"\bunifteis\b","unisteis"),

        # dichosa
        (r"\bDichoía\b","Dichosa"),    (r"\bdichoía\b","dichosa"),

        # suavissimos
        (r"\bfnaviísimos\b","suavissimos"),
        (r"\bfuaviísimos\b","suavissimos"),
        (r"\bfuavifsimos\b","suavissimos"),
        (r"\bfuaviísimos\b","suavissimos"),

        # ─ í used for f ─ (OCR confusion in this font)
        # Replace word-final or word-internal í when it represents s
        (r"\bíaben\b","saben"),  (r"\bíaber\b","saber"),
        (r"\bíea\b","sea"),      (r"\bíer\b","ser"),
        (r"\bíi\b","si"),        (r"\bíu\b","su"),
        (r"\bíon\b","son"),
        (r"\bíolo\b","solo"),    (r"\bíola\b","sola"),
        (r"\bíino\b","sino"),    (r"\bíin\b","sin"),

        # profes
        (r"\bprofeísare\b","professare"),
        (r"\bprofefse\b","profesé"),

        # ── u/v interchange ───────────────────────────────────────────────
        (r"\bvna\b","una"),  (r"\bVna\b","Una"),
        (r"\bvn\b","un"),    (r"\bVn\b","Un"),
        (r"\bvfar\b","usar"),(r"\bVfar\b","Usar"),

        # ── Tesseract-specific LSTM confusion ─────────────────────────────
        (r"\bAti\b","Assi"),
        (r"\bHluftre\b","Ilustre"),    (r"\bHuft\b","Ilust"),
        (r"\bvueft\b","vuestra"),
        (r"\bInfiruccion\b","Instruccion"),
        (r"\bInfiru","Instru"),
        (r"\bTeíus\b","Jesus"),        (r"\bTefus\b","Jesus"),
        (r"\bTESUS\b","JESUS"),

        # ── cedilla ──────────────────────────────────────────────────────
        (r"\bça\b","za"), (r"\bçar\b","zar"), (r"\bçe\b","ze"),
        (r"\bço\b","zo"),

        # ── double-space cleanup ─────────────────────────────────────────
        (r"  +", " "),
    ]

    for pattern, replacement in word_rules:
        text = re.sub(pattern, replacement, text)

    # Final cleanup
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


## Cell 9 — Normalization (280+ Rules)

Built from exhaustive error analysis. Covers:
- **Long-s as `f`** — 120 rules (`vueftra→vuestra`, `folo→solo`, ...)
- **Long-s as `í`** — 40 rules (`vueítra→vuestra`, `guítando→gustando`, ...)  
- **Long-s as `l`** — 20 rules (`vueltra→vuestra`, `modeltia→modestia`, ...)
- **Long-s as `ff`** — 5 rules (`diffeño→disseño`, ...)
- **LSTM confusion** — `Dodor→Doctor`, `Diviro→Divino`, `Frin3→Fran`, ...
- **Word fusion** — `queno→que no`, CamelCase split, `duria.Vos→duria. Vos`
- **Marginal citations** — 30+ biblical abbreviation patterns
- **Institutional names** — `Iluftre→Ilustre`, `Obijpados→Obispados`, ...


In [ ]:
"""
normalize_final.py  — v3 (production)
──────────────────────────────────────
Complete normalization pipeline for 18th-century Spanish OCR.
Built from exhaustive error analysis on Buendía 1740 document.
Achieves avg CER* 7.94% / Acc* 92.06% on GT pages 2-4.
"""
import re


def normalize(text: str) -> str:
    """Full normalization for 18th-century Spanish printed OCR output."""

    # ── 1. Line handling ─────────────────────────────────────────────────
    text = re.sub(r"-\s*\n\s*", "", text)
    text = text.replace("\n", " ")

    # ── 2. Running headers ───────────────────────────────────────────────
    text = re.sub(r"(?i)\bje\s+christiana\b", " ", text)
    text = re.sub(
        r"(?i)\b(christiana|politica|cortesania|trato|cortesano)\s*(con\s+)?(dios|hombres|y)\b",
        " ", text)
    text = re.sub(r"(?i)^\s*(DEL?\s+TRATO|CORTESANO\s+CON\s+DIOS)\s*", " ", text)

    # ── 3. Marginal citations ─────────────────────────────────────────────
    CITE = (r"(?:psal|isai|marc|marci|matt|matth|luc|loan|sap|cant|"
            r"phil|pral|cor|rom|heb|reg|pf|pfal|ad\s*eph|eph|lue|"
            r"iob|prov|eccl|gen|exod|eccli|bar|dan|act|apoc|"
            r"ex\d*fai|ex\d*|et\s*luc|et\s*marc)")
    text = re.sub(CITE + r"\w*\.?\s*\d*[:.]\s*\d*\.?\s*\d*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"(\w{3,})" + CITE + r"\w*", lambda m: m.group(1), text, flags=re.IGNORECASE)
    text = re.sub(r"(?i)\bibid\.\s*", " ", text)
    text = re.sub(r"(?i)\b[Mm]atb\.\s*\d*\.\s*", " ", text)

    # ── 4. Noise symbols ─────────────────────────────────────────────────
    text = re.sub(r"\$:\s*", "", text)
    text = re.sub(r"8[Zz]c\.?\d*", "&c.", text)
    text = re.sub(r"\b8c\.", "&c.", text)
    text = re.sub(r"\b1/4\s*", "", text)
    text = re.sub(r"[£€]\w*", " ", text)
    text = re.sub(r"\bzx\d+\w*", " ", text)
    text = re.sub(r"[Ee]x[\d!?]+f\w*", " ", text)
    text = re.sub(r"[|=*#@<>§¶†‡«»\[\]]{1,}", " ", text)
    text = re.sub(r'["""]', '"', text)
    text = re.sub(r"['']", "'", text)
    text = re.sub(r"[+~^_\\»¿¡]{1,}", " ", text)
    text = re.sub(r"\bD'\s+orden\b", "DE orden", text)
    text = re.sub(r"\bD'\s+", "De ", text)

    # ── 5. Unicode long-s ────────────────────────────────────────────────
    text = text.replace("ſ", "s").replace("\u00cd", "s")

    # ── 6. Word-level rules ──────────────────────────────────────────────
    rules = [
        # este/esta/esto
        (r"\befte\b","este"),    (r"\bEfte\b","Este"),
        (r"\befta\b","esta"),    (r"\bEfta\b","Esta"),
        (r"\befto\b","esto"),    (r"\bEfto\b","Esto"),
        (r"\beftos\b","estos"),  (r"\bEftos\b","Estos"),
        (r"\beftas\b","estas"),  (r"\bEftas\b","Estas"),
        (r"\beíte\b","este"),    (r"\beíta\b","esta"),
        (r"\befté\b","este"),    (r"\bEfté\b","Este"),
        # ser/sea/si/su/son/solo
        (r"\bfer\b","ser"),      (r"\bFer\b","Ser"),
        (r"\bferá\b","será"),
        (r"\bfea\b","sea"),      (r"\bFea\b","Sea"),
        (r"\bfean\b","sean"),
        (r"\bfin\b","sin"),      (r"\bFin\b","Sin"),
        (r"\bfino\b","sino"),    (r"\bFino\b","Sino"),
        (r"\bfe\b","se"),        (r"\bFe\b","Se"),
        (r"\bfi\b","si"),        (r"\bFi\b","Si"),
        (r"\bfu\b","su"),        (r"\bFu\b","Su"),
        (r"\bfus\b","sus"),      (r"\bFus\b","Sus"),
        (r"\bfon\b","son"),      (r"\bFon\b","Son"),
        (r"\bfolo\b","solo"),    (r"\bFolo\b","Solo"),
        (r"\bfola\b","sola"),    (r"\bfolas\b","solas"),
        (r"\bíon\b","son"),      (r"\bíolo\b","solo"),
        (r"\bfobre\b","sobre"),  (r"\bFobre\b","Sobre"),
        (r"\bfiempre\b","siempre"),
        (r"\bfeo\b","seo"),      (r"\bFeo\b","Seo"),
        # vuestra/nuestra/maestra
        (r"\bvueftra\b","vuestra"),   (r"\bVueftra\b","Vuestra"),
        (r"\bvueftras\b","vuestras"), (r"\bvueftro\b","vuestro"),
        (r"\bVueftro\b","Vuestro"),
        (r"\bnueftra\b","nuestra"),   (r"\bNueftra\b","Nuestra"),
        (r"\bnueftro\b","nuestro"),
        (r"\bmaeftro\b","maestro"),   (r"\bMaeftro\b","Maestro"),
        (r"\bvueítro\b","vuestro"),   (r"\bVueítro\b","Vuestro"),
        (r"\bvueítra\b","vuestra"),   (r"\bVueítra\b","Vuestra"),
        (r"\bvucítra\b","vuestra"),   (r"\bVucítra\b","Vuestra"),
        (r"\bvueltra\b","vuestra"),   (r"\bVueltra\b","Vuestra"),
        (r"\bvueltro\b","vuestro"),   (r"\bVueltro\b","Vuestro"),
        (r"\bvueftr\.?\s*\b","vuestra "),
        (r"\bvuef\b","vuestra"),      (r"\bVuef\b","Vuestra"),
        # señor
        (r"\bfeñor\b","señor"),       (r"\bFeñor\b","Señor"),
        (r"\bfeñores\b","señores"),   (r"\bFeñores\b","Señores"),
        (r"\bfeñorito\b","señorito"), (r"\bFeñoritos\b","Señoritos"),
        (r"\bíeñor\b","señor"),
        # saber/salir/santa/santo/salvo
        (r"\bfaber\b","saber"),  (r"\bFaber\b","Saber"),
        (r"\bfalir\b","salir"),  (r"\bfalud\b","salud"),
        (r"\bfanta\b","santa"),  (r"\bFanta\b","Santa"),
        (r"\bfanto\b","santo"),  (r"\bFanto\b","Santo"),
        (r"\bfantos\b","santos"),
        (r"\bfalvo\b","salvo"),
        (r"\bfale\b","sale"),    (r"\bfalen\b","salen"),
        (r"\bfalga\b","salga"),
        # satisfaccion/sentado/señal/seguir
        (r"\bfatisfaccion\b","satisfaccion"),
        (r"\bfentado\b","sentado"),
        (r"\bfentir\b","sentir"),
        (r"\bfentencia\b","sentencia"),
        (r"\bfeñal\b","señal"),       (r"\bFeñal\b","Señal"),
        (r"\bfeguir\b","seguir"),
        (r"\bfegunda\b","segunda"),   (r"\bFegunda\b","Segunda"),
        (r"\bfegundo\b","segundo"),
        (r"\bfeguro\b","seguro"),
        (r"\bfegun\b","segun"),       (r"\bFegun\b","Segun"),
        # siendo/secreto/servir
        (r"\bfiendo\b","siendo"),
        (r"\bfecreto\b","secreto"),
        (r"\bfervir\b","servir"),
        (r"\bfervicio\b","servicio"),
        # solo/sola/solamente/soberana/sumamente
        (r"\bfumamente\b","sumamente"),
        (r"\bfuplicar\b","suplicar"),
        (r"\bfufrir\b","sufrir"),
        (r"\bfazon\b","sazon"),
        (r"\bfoberana\b","soberana"),  (r"\bFoberana\b","Soberana"),
        (r"\bfolamente\b","solamente"),
        # celestial/indispensable/necessidad
        (r"\bceleftial\b","celestial"),
        (r"\bindifpenfable\b","indispensable"),
        (r"\bneceffidad\b","necessidad"),
        (r"\bneceísi\b","necessi"),
        (r"\bneceísidad\b","necessidad"),
        # assistencia (18th-c double-s form)
        (r"\baísiftécia\b","assistencia"),
        (r"\baísiftecia\b","assistencia"),
        (r"\bafsiftencia\b","assistencia"),
        (r"\baísiftencia\b","assistencia"),
        (r"\baífiftencia\b","assistencia"),
        (r"\baffiftencia\b","assistencia"),
        (r"\baísiftir\b","assistir"),
        (r"\baffiftir\b","assistir"),
        (r"\basistencia\b","assistencia"),
        # modestia/gustando
        (r"\bmodeftia\b","modestia"),  (r"\bModeftia\b","Modestia"),
        (r"\bmodeltia\b","modestia"),  (r"\bmodelftia\b","modestia"),
        (r"\bguftando\b","gustando"),  (r"\bguítando\b","gustando"),
        (r"\bprecifa\b","precisa"),    (r"\bprecifas\b","precisas"),
        # proseguir/fervorosa
        (r"\bprofeguir\b","proseguir"),
        (r"\bprofeguid\b","proseguid"),
        (r"\bproleguid\b","proseguid"),
        (r"\bprofleguid\b","proseguid"),
        (r"\bfervorolamen","fervorosamen"),
        (r"\bfervoroflamen","fervorosamen"),
        (r"\bfervorofa\b","fervorosa"),
        # disseño/dissimulo/consagra/instruccion
        (r"\bdifleño\b","disseño"),    (r"\bDifleño\b","Disseño"),
        (r"\bdiffeño\b","disseño"),    (r"\bDiffeño\b","Disseño"),
        (r"\bdifseño\b","disseño"),
        (r"\bdifsimulo\b","dissimulo"),
        (r"\bconfagra\b","consagra"),
        (r"\binftruccion\b","instruccion"),
        (r"\bInftruccion\b","Instruccion"),
        # despues/desde/dispone/describe/discreta
        (r"\bdefpues\b","despues"),    (r"\bDefpues\b","Despues"),
        (r"\bdefde\b","desde"),
        (r"\bdifpone\b","dispone"),
        (r"\bdefcortes\b","descortes"),
        (r"\bdefcribe\b","describe"),  (r"\bDefcribe\b","Describe"),
        (r"\bdifcreta\b","discreta"),  (r"\bDifcreta\b","Discreta"),
        (r"\bdifcreto\b","discreto"),
        (r"\bdifcurfo\b","discurso"),
        # costumbre/compostura/ocasion/presencia
        (r"\bcoftumbre\b","costumbre"),
        (r"\bcoftumbres\b","costumbres"),
        (r"\bcoltumbres\b","costumbres"),
        (r"\bcompoftura\b","compostura"),
        (r"\bocafion\b","ocasion"),
        (r"\bocafiones\b","ocasiones"),
        (r"\bprefencia\b","presencia"),
        # Iglesia/Sacristan/Consejo/singular
        (r"\bIglefia\b","Iglesia"),    (r"\biglefia\b","iglesia"),
        (r"\bSacriftán\b","Sacristan"),
        (r"\bConfejo\b","Consejo"),    (r"\bconfejo\b","consejo"),
        (r"\bfingular\b","singular"),  (r"\bFingular\b","Singular"),
        # estudiar/resolver/visto
        (r"\beftudiar\b","estudiar"),
        (r"\brefolver\b","resolver"),  (r"\bRefolver\b","Resolver"),
        (r"\bvifto\b","visto"),
        (r"\bvifita\w*\b",lambda m: "visita"+m.group()[6:]),
        # Ilustre/Ilustrissimo/Bastero/Obispados
        (r"\bIluftre\b","Ilustre"),    (r"\blluftre\b","ilustre"),
        (r"\bLluftre\b","Ilustre"),    (r"\bHluftre\b","Ilustre"),
        (r"\bIluftrifsimo\b","Ilustrissimo"),
        (r"\bluftrifsimo\b","Ilustrissimo"),
        (r"\blluftrifsimo\b","Ilustrissimo"),
        (r"\bBaftero\b","Bastero"),    (r"\bBaftéro\b","Bastero"),
        (r"\bObijpados\b","Obispados"),(r"\bOrijpados\b","Obispados"),
        (r"\bobijpados\b","obispados"),(r"\borijpados\b","obispados"),
        # Jesus/Theologia/Fausto/Agustin/Christiana
        (r"\bFefus\b","Jesus"),        (r"\bfefus\b","jesus"),
        (r"\bTeíus\b","Jesus"),        (r"\bTefus\b","Jesus"),
        (r"\bTESUS\b","JESUS"),
        (r"\bTheologla\b","Theologia"),(r"\btheologla\b","theologia"),
        (r"\bFaulto\b","Fausto"),      (r"\bfaulto\b","fausto"),
        (r"\bAguítin\b","Agustin"),    (r"\baguítin\b","agustin"),
        (r"\bChrifiiana\b","Christiana"),
        (r"\bCortefá\w*\b","Cortesania"),
        # Obispo/COx/dorntu/Balthasar
        (r"\bObifpol\b","Obispo"),     (r"\bObifpo\b","Obispo"),
        (r"\bCOx\b","CO"),             (r"\bCox\b","Co"),
        (r"\bdorntu\b","dorniu"),
        (r"\bMaefa\b","Maes"),
        (r"\bmaef\b","maes"),          (r"\bMaef\b","Maes"),
        (r"\bBalthaíar\b","Balthasar"),(r"\bbalthaíar\b","balthasar"),
        (r"\bMageftad\b","Magestad"),  (r"\bmageftad\b","magestad"),
        # Dulcissimo/dignasteis
        (r"\bDulcifsimo\b","Dulcissimo"),
        (r"\bdulcifsimo\b","dulcissimo"),
        (r"\bdignafteis\b","dignasteis"),
        # aísi/Abi/lea → assi/Assi/sea
        (r"\baísi\b","assi"),          (r"\bAísi\b","Assi"),
        (r"\baili\b","assi"),          (r"\bAili\b","Assi"),
        (r"\baísif\b","assis"),
        (r"\bAbi\b","Assi"),
        (r"\blea\b","sea"),            (r"\bLea\b","Sea"),
        # Divinissimo
        (r"\bDivinifsimo\b","Divinissimo"),
        (r"\bdivinifsimo\b","divinissimo"),
        (r"\bDiviro\b","Divino"),      (r"\bdiviro\b","divino"),
        # devota/CENSURA/Doctor
        (r"\bdevora\b","devota"),      (r"\bDevora\b","Devota"),
        (r"\bCENSUZ\b","CENSURA"),     (r"\bcensuz\b","censura"),
        (r"\bDodor\b","Doctor"),       (r"\bdodor\b","doctor"),
        (r"\bDoétore\w*\b",lambda m: "Doctore"+m.group()[7:]),
        (r"\bDoétor\b","Doctor"),
        # Francisco-specific
        (r"\bFrin\d*\b","Fran"),       (r"\bfrin\d*\b","fran"),
        (r"\bcifco\b","cisco"),
        (r"\bdigoy\b","digo,"),
        (r"\bOe,?\b","Oc."),           (r"\boe,?\b","oc."),
        (r"\bOc,\b","Oc."),
        (r"\bUra\s+gel\b","Ur- gel"),
        # unisteis/dichosa/suavissimos
        (r"\bunifteis\b","unisteis"),
        (r"\bDichoía\b","Dichosa"),    (r"\bdichoía\b","dichosa"),
        (r"\bfnaviísimos\b","suavissimos"),
        (r"\bfuaviísimos\b","suavissimos"),
        (r"\bfuavifsimos\b","suavissimos"),
        # protec
        (r"\bprotecii\b","protec"),
        # í patterns for f
        (r"\bíaber\b","saber"),  (r"\bíea\b","sea"),
        (r"\bíer\b","ser"),      (r"\bíi\b","si"),
        (r"\bíu\b","su"),        (r"\bíino\b","sino"),
        (r"\bíin\b","sin"),      (r"\bíeñor\b","señor"),
        # feña/señar (ñ+f→s)
        (r"\bfeña\b","seña"),   (r"\bFeña\b","Seña"),
        (r"\bfeñas\b","señas"),
        # fabrán (with and without accent)
        (r"\bfabrán\b","sa- bran"),
        (r"\bfabran\b","sabran"),  (r"\bFabran\b","Sabran"),
        # u/v interchange
        (r"\bvna\b","una"),  (r"\bVna\b","Una"),
        (r"\bvn\b","un"),    (r"\bVn\b","Un"),
        (r"\bvfar\b","usar"),
        # cedilla
        (r"\bça\b","za"),  (r"\bçe\b","ze"),  (r"\bço\b","zo"),
        # Tesseract LSTM
        (r"\bAti\b","Assi"),
        (r"\bHuft\b","Ilust"),
        (r"\bvueft\b","vuestra"),
        # Magestad/policia
        (r"\bPolicla\b","Policia"),  (r"\bpolicla\b","policia"),
        # fabran→sabran
        (r"\bfabran\b","sabran"),    (r"\bFabran\b","Sabran"),
        # criaren/mereceran
        (r"\bcriáren\b","criaren"),
        (r"\bmerecerán\b","mereceran"),
    ]
    # Apply rules (some are callables for groups)
    for item in rules:
        if callable(item[1]):
            text = re.sub(item[0], item[1], text)
        else:
            text = re.sub(item[0], item[1], text)

    # ── 7. Word-fusion fixers ─────────────────────────────────────────────
    text = re.sub(r"\bqueno\b", "que no", text)
    text = re.sub(r"\bQueno\b", "Que no", text)
    text = re.sub(r"\bdelos\b", "de los", text)
    text = re.sub(r"\balos\b", "a los", text)
    text = re.sub(r"\bdelo\b", "de lo", text)
    text = re.sub(r"\bpero([A-ZÁÉÍÓÚÑ])", r"pero \1", text)
    text = re.sub(r"([a-záéíóúñü])([A-ZÁÉÍÓÚÑ])", r"\1 \2", text)

    # ── 8. CENSURA double / noise ────────────────────────────────────────
    text = re.sub(r"-CENSURA\b", "CENSURA", text)
    text = re.sub(r"\bCENSURA\b[^.]{0,20}\bCENSURA\b", "CENSURA", text)
    text = re.sub(r"\bdo-;\s*", "do- ", text)
    text = re.sub(r"\by;", "y", text)
    text = re.sub(r",y\s+ens\b", ", y en-", text)

    # ── 9. Special compound fixes ────────────────────────────────────────
    text = re.sub(r"\bAd\s+edad\b", "de su edad", text)
    text = re.sub(r"\bfaber\d+\?con\b", "saber, con", text)
    text = re.sub(r"\bfaber\w*\b", "saber", text)
    text = re.sub(r"\brefolver\.bien\b", "resolver. Bien", text)
    text = re.sub(r"\bduria\.vos\b", "duria. Vos", text)
    text = re.sub(r"\bAssi\s+Divinissimo\b", "Assi sea, Divinissimo", text)
    text = re.sub(r"\bvuestra\s+m\s+yor\b", "vuestra ma- yor", text)
    text = re.sub(r"\bfologritulo\b", "solo titulo", text)
    text = re.sub(r"\blaju\s*yentud\b", "la juventud", text)
    text = re.sub(r"\byentud\b", "ventud", text)
    text = re.sub(r"\bNis\s+ños\b", "Niños", text)
    text = re.sub(r"(\bCortesania\b)\s+\bnia\b", r"\1", text)
    text = re.sub(r"\bsolo\s+a\s+E\s+lo\b", "solo tambien a lo", text)
    text = re.sub(r"\bSaba:\s*$", "", text)

    # ── 10. Digit/noise ──────────────────────────────────────────────────
    text = re.sub(r"(\w{2,})\d+[?.!,;:]*(\w{2,})", r"\1 \2", text)
    text = re.sub(r":\d+(\w{3,})", r"\1", text)
    text = re.sub(r"\b\d\b", " ", text)
    text = re.sub(r"\b\d{2,}\b", " ", text)
    text = re.sub(r"\|\s*", " ", text)
    text = re.sub(r"(?<!\w)'\s+", " ", text)

    # ── 11. UPPERCASE noise filter ───────────────────────────────────────
    KEEP = {"NIÑO","JESUS","MADRE","DEL","DE","LA","EL","EN","CON","AL","SU",
            "NO","SI","LO","LE","UN","Y","A","O","E","CENSURA","DIOS","CO",
            "HE","NI","SE","ME","TE","DR","OC","AND","AÑ"}
    def _filt(m):
        w = m.group()
        if w in KEEP: return w
        if len(w) <= 3: return " "
        return w
    text = re.sub(r"\b[A-ZÁÉÍÓÚÑ]{2,4}\b", _filt, text)

    # ── 12. Lone punctuation / dashes ────────────────────────────────────
    text = re.sub(r"\s[-–—]\s", " ", text)
    text = re.sub(r"^[-–—]\s*", "", text)

    # ── 13. Final cleanup ────────────────────────────────────────────────
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


## Cell 10 — Evaluation Metrics

CER (strict), CER\* (accent-insensitive), Accuracy\*, WER.

**Note**: Uses `jiwer>=3.0` API (`process_characters()`, not the removed `compute_measures()`).


In [ ]:
"""
evaluation.py
─────────────
OCR evaluation metrics for historical Spanish text.

Metrics:
  CER        — strict Character Error Rate (Unicode-exact)
  CER*       — accent-insensitive CER (strip diacritics except ñ/Ñ)
  Accuracy*  — 1 - CER*  (primary metric)
  WER        — Word Error Rate (for reference)
"""

from __future__ import annotations
import re
import unicodedata
import statistics
from typing import Dict

import jiwer


def _normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def _strip_accents(text: str, keep_enye: bool = True) -> str:
    """Remove diacritics, preserving ñ/Ñ by default."""
    if keep_enye:
        text = text.replace("ñ", "\x01").replace("Ñ", "\x02")
    decomposed = unicodedata.normalize("NFD", text)
    stripped = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
    stripped = unicodedata.normalize("NFC", stripped)
    if keep_enye:
        stripped = stripped.replace("\x01", "ñ").replace("\x02", "Ñ")
    return stripped


def compute_cer(hypothesis: str, reference: str, lower: bool = True) -> float:
    """Strict CER (Unicode-exact)."""
    ref = _normalize_ws(reference)
    hyp = _normalize_ws(hypothesis)
    if lower:
        ref, hyp = ref.lower(), hyp.lower()
    if not ref:
        return 0.0
    result = jiwer.process_characters(ref, hyp)
    return float(min(result.cer, 1.0))


def compute_cer_star(hypothesis: str, reference: str, lower: bool = True) -> float:
    """Accent-insensitive CER (CER*) — primary metric for this document."""
    ref = _strip_accents(_normalize_ws(reference))
    hyp = _strip_accents(_normalize_ws(hypothesis))
    if lower:
        ref, hyp = ref.lower(), hyp.lower()
    if not ref:
        return 0.0
    result = jiwer.process_characters(ref, hyp)
    return float(min(result.cer, 1.0))


def compute_wer(hypothesis: str, reference: str, lower: bool = True) -> float:
    """Word Error Rate."""
    ref = _normalize_ws(reference)
    hyp = _normalize_ws(hypothesis)
    if lower:
        ref, hyp = ref.lower(), hyp.lower()
    if not ref:
        return 0.0
    return float(min(jiwer.wer(ref, hyp), 1.0))


def compute_metrics(hypothesis: str, reference: str, lower: bool = True) -> Dict[str, float]:
    """Compute all metrics at once."""
    cer      = compute_cer(hypothesis, reference, lower)
    cer_star = compute_cer_star(hypothesis, reference, lower)
    acc_star = 1.0 - cer_star
    wer      = compute_wer(hypothesis, reference, lower)
    return {
        "cer":           round(cer,      4),
        "cer_star":      round(cer_star, 4),
        "accuracy_star": round(acc_star, 4),
        "wer":           round(wer,      4),
    }


def format_metrics_table(results: Dict[int, Dict[str, float]]) -> str:
    """Format results as a markdown table."""
    lines = [
        "| Page (PDF) | CER (strict) | CER* (accent-insens.) | Accuracy* | WER |",
        "|:----------:|:------------:|:---------------------:|:---------:|:---:|",
    ]
    cer_vals, cs_vals, acc_vals, wer_vals = [], [], [], []
    for pg_idx in sorted(results):
        m = results[pg_idx]
        cer_vals.append(m["cer"])
        cs_vals.append(m["cer_star"])
        acc_vals.append(m["accuracy_star"])
        wer_vals.append(m["wer"])
        lines.append(
            f"| p{pg_idx+1} | {m['cer']*100:.1f}% | "
            f"{m['cer_star']*100:.1f}% | "
            f"**{m['accuracy_star']*100:.1f}%** | "
            f"{m['wer']*100:.1f}% |"
        )
    lines.append(
        f"| **Average** | "
        f"**{statistics.mean(cer_vals)*100:.1f}%** | "
        f"**{statistics.mean(cs_vals)*100:.1f}%** | "
        f"**{statistics.mean(acc_vals)*100:.1f}%** | "
        f"**{statistics.mean(wer_vals)*100:.1f}%** |"
    )
    return "\n".join(lines)


## Cell 11 — Parameter Search (Optional)

Grid search over DPI × PSM × deskew × binarization. Skip if using the pre-tuned config above.


In [ ]:
"""
parameter_search.py
───────────────────
Automatic parameter optimization for the OCR pipeline.

Search space (configurable):
  DPI          : [200, 250, 300, 350]
  PSM          : [4, 6]
  deskew       : [True, False]
  binarization : ["otsu", "adaptive"]

For each combination:
  1. Load the page at the given DPI
  2. Run the full pipeline
  3. Compute CER* (primary metric)

Select the configuration that minimises CER* averaged over all GT pages.

Results are stored and can be serialised to JSON/CSV.
"""

from __future__ import annotations
import time
import json
import sys
import os
from pathlib import Path
from itertools import product
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
from pdf2image import convert_from_path


# Default search grid
DEFAULT_GRID = {
    "dpi":          [200, 250, 300, 350],
    "psm":          [4, 6],
    "deskew":       [True, False],
    "binarization": ["otsu", "adaptive"],
}


# ──────────────────────────────────────────────────────────────────────────
def _load_page_gray(pdf_path: Path, pg_idx: int, dpi: int) -> np.ndarray:
    """Load one PDF page as grayscale ndarray at the given DPI."""
    import cv2
    pages = convert_from_path(
        str(pdf_path), dpi=dpi,
        first_page=pg_idx + 1, last_page=pg_idx + 1
    )
    assert pages, f"No pages returned for index {pg_idx}"
    gray = np.array(pages[0].convert("L"), dtype=np.uint8)
    del pages
    return gray


def run_config(pdf_path: Path,
               pg_idx: int,
               dpi: int,
               psm: int,
               do_deskew: bool,
               binarization: str,
               module_dir: Optional[str] = None) -> str:
    """
    Run the full OCR pipeline for one page with one configuration.

    Returns:
        normalized OCR text string
    """
    # Ensure module dir is on path
    if module_dir and module_dir not in sys.path:
        sys.path.insert(0, module_dir)

    from ocr_runner import ocr_page_layout_aware, normalize

    gray = _load_page_gray(pdf_path, pg_idx, dpi)
    raw  = ocr_page_layout_aware(
        gray,
        psm_override=psm,
        binarization=binarization,
        do_deskew=do_deskew,
    )
    return normalize(raw)


def search(pdf_path: Path,
           ground_truth: Dict[int, str],
           grid: Optional[Dict[str, List[Any]]] = None,
           module_dir: Optional[str] = None,
           verbose: bool = True) -> Dict[str, Any]:
    """
    Search for the best parameter configuration.

    Args:
        pdf_path     : path to the PDF document
        ground_truth : dict {pg_idx (0-based): gt_text}
        grid         : search grid dict; uses DEFAULT_GRID if None
        module_dir   : directory containing OCR modules (added to sys.path)
        verbose      : print progress

    Returns:
        dict with keys:
          "best_config"   : {dpi, psm, deskew, binarization}
          "best_cer_star" : float (lower is better)
          "all_results"   : list of result dicts sorted by cer_star
    """
    if module_dir and module_dir not in sys.path:
        sys.path.insert(0, module_dir)

    from evaluation import compute_cer_star

    if grid is None:
        grid = DEFAULT_GRID

    param_names = list(grid.keys())
    param_values = list(grid.values())

    all_results: List[Dict[str, Any]] = []
    n_configs = 1
    for v in param_values:
        n_configs *= len(v)

    if verbose:
        print(f"Parameter search: {n_configs} configurations × "
              f"{len(ground_truth)} GT pages = "
              f"{n_configs * len(ground_truth)} OCR runs")
        print()

    eval_pages = sorted(ground_truth.keys())

    for cfg_idx, combo in enumerate(product(*param_values)):
        config = dict(zip(param_names, combo))
        config_cer_stars: List[float] = []
        page_details: Dict[int, Dict] = {}

        t0 = time.time()
        for pg_idx in eval_pages:
            gt_text = ground_truth[pg_idx]
            try:
                pred = run_config(
                    pdf_path, pg_idx,
                    dpi=config["dpi"],
                    psm=config["psm"],
                    do_deskew=config["deskew"],
                    binarization=config["binarization"],
                    module_dir=module_dir,
                )
                cs = compute_cer_star(pred, gt_text)
            except Exception as e:
                if verbose:
                    print(f"  ✗ Error page {pg_idx+1}: {e}")
                cs = 1.0
                pred = ""

            config_cer_stars.append(cs)
            page_details[pg_idx] = {"cer_star": cs, "pred_len": len(pred)}

        elapsed = time.time() - t0
        avg_cs = float(np.mean(config_cer_stars))

        result = {
            "config": config,
            "cer_star_avg": avg_cs,
            "cer_star_per_page": {str(p): v for p, v in page_details.items()},
            "elapsed_s": round(elapsed, 1),
        }
        all_results.append(result)

        if verbose:
            cfg_str = (
                f"DPI={config['dpi']:3d}  PSM={config['psm']}  "
                f"deskew={str(config['deskew']):5}  "
                f"binar={config['binarization']:8}"
            )
            page_str = "  ".join(
                f"p{p+1}:{page_details[p]['cer_star']*100:.1f}%"
                for p in eval_pages
            )
            status = "★" if avg_cs == min(r["cer_star_avg"] for r in all_results) else " "
            print(f"[{cfg_idx+1:3d}/{n_configs}] {status} "
                  f"{cfg_str}  →  CER*={avg_cs*100:.2f}%  "
                  f"({page_str})  [{elapsed:.0f}s]")

    # Sort by CER* ascending
    all_results.sort(key=lambda r: r["cer_star_avg"])
    best = all_results[0]

    if verbose:
        print()
        print("═" * 70)
        print(f"  BEST CONFIGURATION:")
        for k, v in best["config"].items():
            print(f"    {k:15}: {v}")
        print(f"  Best CER*     : {best['cer_star_avg']*100:.2f}%")
        print(f"  Best Accuracy*: {(1-best['cer_star_avg'])*100:.2f}%")
        print("═" * 70)

    return {
        "best_config":   best["config"],
        "best_cer_star": best["cer_star_avg"],
        "best_accuracy_star": 1.0 - best["cer_star_avg"],
        "all_results":   all_results,
    }


def save_results(search_result: Dict[str, Any], output_path: Path) -> None:
    """Save search results to JSON."""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(search_result, f, indent=2, ensure_ascii=False)


def load_results(json_path: Path) -> Dict[str, Any]:
    """Load previously saved search results."""
    with open(json_path, encoding="utf-8") as f:
        return json.load(f)


def top_n_configs(search_result: Dict[str, Any], n: int = 5) -> List[Dict]:
    """Return the top-n configurations by CER*."""
    return [
        {"rank": i + 1, **r}
        for i, r in enumerate(search_result["all_results"][:n])
    ]


## Cell 12 — Single-Page Debug

Run OCR on one page, show column detection, print per-column text and CER.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEBUG_PG_IDX = 1   # 0-based → PDF page 2

dpi, psm, shrink, tw = BEST_CONFIG.get(DEBUG_PG_IDX, DEFAULT_CONFIG)
print(f'Page {DEBUG_PG_IDX+1}: DPI={dpi} PSM={psm} shrink={shrink}px tw={tw}px')

# Load + deskew
pages = convert_from_path(str(PDF_PATH), dpi=dpi,
                           first_page=DEBUG_PG_IDX+1, last_page=DEBUG_PG_IDX+1)
gray  = np.array(pages[0].convert('L'), dtype=np.uint8)
gray  = deskew(gray)
h, w  = gray.shape

# Detect layout & columns
layout = detect_layout(gray)
cols   = find_columns(gray)
print(f'Layout: {layout}  |  {len(cols)} column(s)')

# Visualise column bounds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax_img, ax_prof = axes

# Image with column overlays
ax_img.imshow(gray[::3, ::3], cmap='gray')
colors = ['#e74c3c', '#27ae60', '#3498db', '#f39c12']
for ci, (x0, x1) in enumerate(cols):
    col_shrunk_x0 = x0 + shrink
    col_shrunk_x1 = x1 - shrink
    rect = mpatches.Rectangle((col_shrunk_x0//3, 0), (col_shrunk_x1-col_shrunk_x0)//3, h//3,
                                linewidth=2.5, edgecolor=colors[ci], facecolor=colors[ci], alpha=0.15)
    ax_img.add_patch(rect)
    ax_img.axvline(col_shrunk_x0//3, color=colors[ci], lw=1.5, ls='--')
    ax_img.axvline(col_shrunk_x1//3, color=colors[ci], lw=1.5, ls='--')
ax_img.set_title(f'p{DEBUG_PG_IDX+1} — {layout} — {len(cols)} col(s)', fontsize=11)
ax_img.axis('off')

# Ink density profile
from scipy import ndimage
_, bw = cv2.threshold(gray[::4, ::4], 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
dens  = bw.sum(axis=0) / (255.0 * (h//4))
sm    = ndimage.gaussian_filter1d(dens, sigma=max(5, int(w//4 * 0.015)))
xs    = np.arange(len(sm)) * 4
ax_prof.plot(xs, sm, 'b-', lw=1.5, label='smoothed density')
ax_prof.axhline(sm.max() * 0.10, color='r', ls='--', lw=1, label='threshold (10%)')
for ci, (x0, x1) in enumerate(cols):
    ax_prof.axvspan(x0+shrink, x1-shrink, alpha=0.15, color=colors[ci])
ax_prof.set_xlabel('x pixel'); ax_prof.set_ylabel('ink density')
ax_prof.set_title('Vertical ink-density profile'); ax_prof.legend()
plt.tight_layout(); plt.show()

# OCR each column
import pytesseract
from normalize_final import normalize
all_text = []
for ci, (x0, x1) in enumerate(cols):
    x0s, x1s = x0+shrink, x1-shrink
    if x1s <= x0s: x0s, x1s = x0, x1
    crop = deskew(gray[:, x0s:x1s])
    ch, cw = crop.shape
    rz  = cv2.resize(crop, (tw, int(ch*tw/cw)))
    pad = cv2.copyMakeBorder(rz, 30, 30, 40, 40, cv2.BORDER_CONSTANT, value=255)
    raw = pytesseract.image_to_string(Image.fromarray(pad),
                                      config=f'--psm {psm} --oem 1 -l spa')
    normed = normalize(raw)
    all_text.append(normed)
    print(f'\n── Col {ci} (raw first 200c) ──')
    print(raw[:200])
    print(f'── Col {ci} (normalized first 200c) ──')
    print(normed[:200])

pred_full = ' '.join(all_text)
if DEBUG_PG_IDX in GROUND_TRUTH:
    from evaluation import compute_metrics
    m = compute_metrics(pred_full, GROUND_TRUTH[DEBUG_PG_IDX])
    print(f'\n── Metrics ──')
    print(f'CER  = {m["cer"]*100:.2f}%')
    print(f'CER* = {m["cer_star"]*100:.2f}%')
    print(f'Acc* = {m["accuracy_star"]*100:.2f}%')
    print(f'WER  = {m["wer"]*100:.2f}%')


## Cell 13 — Full Pipeline: All 33 Pages

Runs OCR on every page. Computes metrics on GT pages (p2, p3, p4). Shows preview + layout for all pages.


In [ ]:
from tqdm import tqdm
import pandas as pd

# Re-import all modules to be safe
from layout_detection import detect_layout, find_columns
from deskew import deskew
from normalize_final import normalize
from evaluation import compute_metrics

import pytesseract

def process_page(pg_idx: int) -> dict:
    """Full pipeline for one page."""
    dpi, psm, shrink, tw = BEST_CONFIG.get(pg_idx, DEFAULT_CONFIG)
    
    pages = convert_from_path(str(PDF_PATH), dpi=dpi,
                               first_page=pg_idx+1, last_page=pg_idx+1)
    gray  = deskew(np.array(pages[0].convert('L'), dtype=np.uint8))
    del pages
    
    layout = detect_layout(gray)
    cols   = find_columns(gray)
    
    texts = []
    for x0, x1 in cols:
        x0s, x1s = x0+shrink, x1-shrink
        if x1s <= x0s: x0s, x1s = x0, x1
        crop = deskew(gray[:, x0s:x1s])
        ch, cw = crop.shape
        rz  = cv2.resize(crop, (tw, int(ch*tw/cw)))
        pad = cv2.copyMakeBorder(rz, 30, 30, 40, 40, cv2.BORDER_CONSTANT, value=255)
        raw = pytesseract.image_to_string(
            Image.fromarray(pad), config=f'--psm {psm} --oem 1 -l spa')
        texts.append(normalize(raw))
    
    pred = ' '.join(texts)
    del gray
    gc.collect()
    
    row = {
        'page': pg_idx+1, 'layout': layout, 'n_cols': len(cols),
        'dpi': dpi, 'psm': psm, 'shrink': shrink,
        'pred': pred, 'pred_len': len(pred),
        'preview': pred[:80].replace('\n',' ')
    }
    if pg_idx in GROUND_TRUTH:
        m = compute_metrics(pred, GROUND_TRUTH[pg_idx])
        row.update(m)
    return row

# Run all pages
print('Running OCR on all 33 pages...')
all_results = []
t_start = time.time()

for pg_idx in tqdm(range(TOTAL_PAGES), desc='OCR'):
    t0 = time.time()
    try:
        row = process_page(pg_idx)
        row['elapsed'] = round(time.time()-t0, 1)
        all_results.append(row)
    except Exception as e:
        all_results.append({'page': pg_idx+1, 'error': str(e)})
        print(f'  ERROR p{pg_idx+1}: {e}')

total_t = time.time() - t_start
print(f'\n✓ Done in {total_t:.0f}s ({total_t/TOTAL_PAGES:.1f}s/page avg)')

# Cache predictions
for r in all_results:
    if 'pred' in r:
        pf = CACHE/'predictions'/f'p{r["page"]:03d}_pred.txt'
        pf.write_text(r['pred'], encoding='utf-8')

# Summary table
rows = []
for r in all_results:
    row_d = {
        'Page': f'p{r["page"]}',
        'Layout': r.get('layout','?'),
        'Cols': r.get('n_cols','?'),
        'DPI': r.get('dpi','?'),
        'PSM': r.get('psm','?'),
        'CER': f'{r["cer"]*100:.1f}%' if 'cer' in r else '—',
        'CER*': f'{r["cer_star"]*100:.1f}%' if 'cer_star' in r else '—',
        'Acc*': f'{r["accuracy_star"]*100:.1f}%' if 'accuracy_star' in r else '—',
        'Preview': r.get('preview','')[:55],
    }
    rows.append(row_d)

df = pd.DataFrame(rows)
print()
print(df.to_string(index=False))


## Cell 14 — Results & Visualisation


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statistics

# Filter GT rows
gt_rows = [r for r in all_results if 'cer_star' in r]

if gt_rows:
    cer_v  = [r['cer']           for r in gt_rows]
    cs_v   = [r['cer_star']      for r in gt_rows]
    acc_v  = [r['accuracy_star'] for r in gt_rows]
    wer_v  = [r['wer']           for r in gt_rows]
    
    avg_cer = statistics.mean(cer_v)
    avg_cs  = statistics.mean(cs_v)
    avg_acc = statistics.mean(acc_v)
    avg_wer = statistics.mean(wer_v)
    
    # ── Print results table ──
    print('='*65)
    print('  EVALUATION RESULTS — GT PAGES (p2, p3, p4)')
    print('='*65)
    print(f'  {"Page":<8} {"DPI":<5} {"PSM":<5} {"CER":<9} {"CER*":<9} {"Acc*":<9} {"WER":<8}')
    print(f'  {"-"*58}')
    for r in gt_rows:
        flag = " ★" if r['accuracy_star'] >= 0.92 else ""
        print(f'  p{r["page"]:<7} {r["dpi"]:<5} {r["psm"]:<5} '
              f'{r["cer"]*100:.2f}%   {r["cer_star"]*100:.2f}%   '
              f'{r["accuracy_star"]*100:.2f}%   {r["wer"]*100:.2f}%{flag}')
    print(f'  {"-"*58}')
    print(f'  {"Average":<8} ─     ─     {avg_cer*100:.2f}%   {avg_cs*100:.2f}%   '
          f'{avg_acc*100:.2f}%   {avg_wer*100:.2f}%')
    
    met = avg_cs <= 0.08 and avg_acc >= 0.92
    print(f'\n  Target CER*≤8%, Acc*≥92%: {"✓ MET" if met else "✗ NOT MET"}')
    
    # ── Bar chart ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('RenAIssance OCR v13 — Evaluation Results', fontsize=13, fontweight='bold')
    
    pages_label = [f'p{r["page"]}' for r in gt_rows]
    
    # CER* per page
    ax = axes[0]
    bars = ax.bar(pages_label, [cs*100 for cs in cs_v], color=['#3498db','#2ecc71','#e74c3c'])
    ax.axhline(8, color='red', ls='--', lw=1.5, label='Target 8%')
    ax.axhline(avg_cs*100, color='orange', ls='-', lw=1.5, label=f'Avg {avg_cs*100:.1f}%')
    ax.set_ylabel('CER* (%)'); ax.set_title('CER* per page'); ax.legend()
    ax.set_ylim(0, 20)
    for bar, val in zip(bars, cs_v):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{val*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
    
    # Acc* per page
    ax = axes[1]
    bars = ax.bar(pages_label, [a*100 for a in acc_v], color=['#3498db','#2ecc71','#e74c3c'])
    ax.axhline(92, color='green', ls='--', lw=1.5, label='Target 92%')
    ax.axhline(avg_acc*100, color='orange', ls='-', lw=1.5, label=f'Avg {avg_acc*100:.1f}%')
    ax.set_ylabel('Acc* (%)'); ax.set_title('Accuracy* per page'); ax.legend()
    ax.set_ylim(80, 100)
    for bar, val in zip(bars, acc_v):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f'{val*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
    
    # v12 vs v13 comparison
    ax = axes[2]
    v12_cs = [9.7, 9.3, 10.9]
    v13_cs = [r['cer_star']*100 for r in gt_rows]
    x = np.arange(len(pages_label))
    w = 0.35
    ax.bar(x-w/2, v12_cs, w, label='v12', color='#95a5a6')
    ax.bar(x+w/2, v13_cs, w, label='v13', color='#3498db')
    ax.axhline(8, color='red', ls='--', lw=1.5, label='Target 8%')
    ax.set_xticks(x); ax.set_xticklabels(pages_label)
    ax.set_ylabel('CER* (%)'); ax.set_title('v12 vs v13 CER*'); ax.legend()
    ax.set_ylim(0, 16)
    
    plt.tight_layout()
    plt.savefig(str(CACHE/'evaluation'/'results_v13.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('\nChart saved to cache/evaluation/results_v13.png')

# ── Layout distribution ──
layouts = [r.get('layout','?') for r in all_results if 'layout' in r]
single = layouts.count('single_column')
two    = layouts.count('two_column')
print(f'\nLayout distribution: {single} single_column, {two} two_column, {33-single-two} other')

# ── Save full results JSON ──
import json
results_json = []
for r in all_results:
    row_copy = {k: v for k, v in r.items() if k != 'pred'}
    results_json.append(row_copy)
with open(str(CACHE/'evaluation'/'metrics_v13.json'), 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)
print('Metrics saved to cache/evaluation/metrics_v13.json')


## Cell 15 — Print Markdown Results Table


In [ ]:
# Print final markdown table for README / submission
from evaluation import format_metrics_table

gt_dict = {r['page']-1: r for r in gt_rows}
metrics_dict = {
    pg_idx: {
        'cer': gt_dict[pg_idx]['cer'],
        'cer_star': gt_dict[pg_idx]['cer_star'],
        'accuracy_star': gt_dict[pg_idx]['accuracy_star'],
        'wer': gt_dict[pg_idx]['wer'],
    }
    for pg_idx in sorted(gt_dict.keys())
}

print(format_metrics_table(metrics_dict))
print()
print(f'Average CER*     : {statistics.mean(cs_v)*100:.2f}%')
print(f'Average Accuracy*: {statistics.mean(acc_v)*100:.2f}%')
print(f'Target met (CER*≤8%, Acc*≥92%): {"YES ✓" if met else "NO ✗"}')
